In [1]:
import json
import os
import subprocess
from pathlib import Path
from typing import Any

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langchain_ollama import ChatOllama



In [14]:
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://10.42.0.247:11434/",
)

MODEL_NAME = os.getenv(
    "OLLAMA_MODEL",
    "gpt-oss:20b",
)

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.1,
    num_ctx=8192,
)

print(f"Using {MODEL_NAME} at {OLLAMA_BASE_URL}")

Using gpt-oss:20b at http://10.42.0.247:11434/


In [15]:
response = llm.invoke("Reply with exactly: Ollama connection works")
print(response.content)


Ollama connection works


In [16]:
WORKSPACE = Path("/workspace/agent_project")
WORKSPACE.mkdir(parents=True, exist_ok=True)

WORKSPACE = WORKSPACE.resolve()
print(WORKSPACE)

/workspace/agent_project


In [18]:
def safe_path(relative_path: str) -> Path:
    """
    Resolve a user-provided path inside WORKSPACE.
    Prevents paths such as ../../etc/passwd.
    """
    path = (WORKSPACE / relative_path).resolve()

    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError(f"Path escapes workspace: {relative_path}")

    return path

In [19]:
@tool
def list_files() -> str:
    """List files and directories in the current project workspace."""
    entries = []

    for path in sorted(WORKSPACE.rglob("*")):
        relative = path.relative_to(WORKSPACE)
        if ".git" in relative.parts or "__pycache__" in relative.parts:
            continue

        suffix = "/" if path.is_dir() else ""
        entries.append(f"{relative}{suffix}")

    return "\n".join(entries) if entries else "(workspace is empty)"


In [20]:
@tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file from the project workspace."""
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    # Avoid flooding the model context with very large files.
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + "\n...[truncated]"

    return content


In [21]:
from importlib.metadata import version

for package in [
    "langchain",
    "langchain-core",
    "langchain-ollama",
]:
    print(package, version(package))

langchain 1.3.16
langchain-core 1.6.0
langchain-ollama 1.1.0


In [22]:
@tool
def write_file(path: str, content: str) -> str:
    """Create or overwrite a UTF-8 text file inside the project workspace."""
    file_path = safe_path(path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding="utf-8")

    return f"Wrote {len(content)} characters to {file_path.relative_to(WORKSPACE)}"


In [23]:
@tool
def run_command(command: str) -> str:
    """
    Run a non-interactive shell command inside the project workspace.
    Use this for formatting, tests, compilation, and inspection.
    """
    blocked_fragments = [
        "rm -rf",
        "shutdown",
        "reboot",
        "mkfs",
        "dd if=",
        ":(){",
        "curl | sh",
        "wget | sh",
    ]

    normalized = command.lower().replace(" ", "")
    for fragment in blocked_fragments:
        if fragment.replace(" ", "") in normalized:
            return f"Blocked potentially destructive command: {command}"

    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=WORKSPACE,
            capture_output=True,
            text=True,
            timeout=60,
            env={
                **os.environ,
                "PYTHONUNBUFFERED": "1",
            },
        )

        output = (
            f"exit_code: {result.returncode}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

        if len(output) > 20_000:
            output = output[:20_000] + "\n...[output truncated]"

        return output

    except subprocess.TimeoutExpired:
        return "Command timed out after 60 seconds."
    except Exception as exc:
        return f"Command failed to run: {type(exc).__name__}: {exc}"


In [24]:
TOOLS = [
    list_files,
    read_file,
    write_file,
    run_command,
]

TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

for item in TOOLS:
    print(item.name)


list_files
read_file
write_file
run_command


In [25]:
llm_with_tools = llm.bind_tools(TOOLS)


In [26]:
response = llm_with_tools.invoke(
    "Use the list_files tool and report the files in the workspace."
)

print("content:", response.content)
print("tool calls:", response.tool_calls)

content: 
tool calls: [{'name': 'list_files', 'args': {}, 'id': 'eb2df8f0-5b8b-40d3-b9b0-7cbba269b00a', 'type': 'tool_call'}]


In [27]:
SYSTEM_PROMPT = """
You are a coding agent working inside a project workspace.

Your job is to turn the user's software requirements into working files.

Rules:
- Inspect existing files before modifying them.
- Use write_file to create or update source files.
- Use read_file to inspect relevant files.
- Use run_command for tests, formatters, linters, compilers, and basic inspection.
- Do not claim that code works unless you actually run an appropriate check.
- Keep generated code focused and maintainable.
- Ask for clarification only when the requirement is genuinely ambiguous.
- Do not delete or overwrite unrelated files.
- All paths must be relative to the project workspace.
"""


In [28]:
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

def run_agent(
    user_request: str,
    max_iterations: int = 12,
    verbose: bool = True,
) -> str:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_request),
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- iteration {iteration + 1} ---")

        response = llm_with_tools.invoke(messages)
        messages.append(response)

        tool_calls = response.tool_calls or []

        if verbose:
            if response.content:
                print("Assistant:", response.content)
            print("Tool calls:", tool_calls)

        # The model is finished when it returns no tool calls.
        if not tool_calls:
            return response.content or "(no final response)"

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call["id"]

            selected_tool = TOOLS_BY_NAME.get(tool_name)

            if selected_tool is None:
                tool_result = f"Unknown tool: {tool_name}"
            else:
                try:
                    tool_result = selected_tool.invoke(tool_args)
                except Exception as exc:
                    tool_result = (
                        f"Tool error: {type(exc).__name__}: {exc}"
                    )

            if verbose:
                print(f"Executing: {tool_name}({tool_args})")
                print(str(tool_result)[:2_000])

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call_id,
                )
            )

    return (
        f"Agent stopped after {max_iterations} iterations. "
        "The workspace may contain partial results."
    )


In [29]:
request = """
Create a small REST API project with:

1. A Python FastAPI service in app/main.py.
2. A TypeScript client in client/index.ts that calls the API.
3. A Go command-line client in cmd/client/main.go.
4. A README.md explaining how to run each component.
5. Basic input validation and a health endpoint.

Use simple in-memory data only. After creating the files, inspect the project
and run syntax or compilation checks that are available in the container.
"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)



--- iteration 1 ---
Tool calls: [{'name': 'list_files', 'args': {}, 'id': '37ab9a86-ba30-4161-8e98-27231ae85028', 'type': 'tool_call'}]
Executing: list_files({})
(workspace is empty)

--- iteration 2 ---
Tool calls: [{'name': 'write_file', 'args': {'path': 'requirements.txt', 'content': 'fastapi\nuvicorn[standard]\n'}, 'id': '653f9fba-a9e7-40af-b497-936656f3618c', 'type': 'tool_call'}]
Executing: write_file({'path': 'requirements.txt', 'content': 'fastapi\nuvicorn[standard]\n'})
Wrote 26 characters to requirements.txt

--- iteration 3 ---
Tool calls: [{'name': 'run_command', 'args': {'command': 'python --version'}, 'id': 'f7ad2de0-885a-4244-b28f-23a3ab12ac68', 'type': 'tool_call'}]
Executing: run_command({'command': 'python --version'})
exit_code: 0
stdout:
Python 3.12.14

stderr:


--- iteration 4 ---
Tool calls: [{'name': 'run_command', 'args': {'command': 'node --version'}, 'id': '2f79b82b-5eb8-4821-a4b0-9c2c5674c276', 'type': 'tool_call'}]
Executing: run_command({'command': 'node 

In [30]:
print(list_files.invoke({}))


README.md
app/
app/main.py
client/
client/index.ts
cmd/
cmd/client/
cmd/client/main.go
requirements.txt


In [ ]:
print(read_file.invoke({"path": "app/main.py"}))


In [31]:
print(run_command.invoke({
    "command": "python -m compileall app"
}))

exit_code: 0
stdout:
Listing 'app'...

stderr:



In [ ]:
def interactive_agent():
    print("Coding agent ready.")
    print(f"Workspace: {WORKSPACE}")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        try:
            request = input("You> ").strip()
        except EOFError:
            break

        if request.lower() in {"exit", "quit"}:
            break

        if not request:
            continue

        answer = run_agent(request, verbose=True)
        print(f"\nAgent> {answer}\n")
